In [ ]:
import numpy as np
from statsmodels.stats.multitest import multipletests

from _load_brain_atlases import load_brain_atlases
from _results_computation import (read_pickle, compare_conditions, threshold_schaefers,
                                  find_max, save_schaefer_maps, plot_shared_voxels)

In [ ]:
# load in records
# comparing recursion and iteration
records = read_pickle("output/organized_results_by_condition.pkl")
nulls = read_pickle('output/null_distributions.pkl')
atlases = load_brain_atlases()

In [ ]:
sig_voxels = {
    'recursion':[],
    'iteration':[],
    'prose'    :[],
    'list'     :[]
}

corr_coefficients = {
    'recursion':[],
    'iteration':[],
    'prose'    :[],
    'list'     :[]
}

all_sigs = []
for i,row in records.iterrows():
    # initially saved the voxels that weren't significant by mistake, so calculating the number of voxels that are
    num_sig_voxels = len(row['sig_voxel_idx'])
    condition = row['condition']
    sig_voxels[condition].append(num_sig_voxels)
    all_sigs.append(num_sig_voxels)

    corr_coefficients[condition].append(row['participant_mean'])

    
for condition,vox in sig_voxels.items():
    print(f"Average number of significant voxels in {condition}: {np.mean(vox):.3f} ($\\sigma={np.std(vox):.3f}$). Average correlation is {np.mean(corr_coefficients[condition])} ($\\sigma={np.std(corr_coefficients[condition])}$)")
print(f"Average number of significant voxels across conditions: {np.mean(all_sigs):.3f} (\\signam{np.std(all_sigs):.3f})")

cutoff = round(np.mean(all_sigs))

In [ ]:
# Shared / unique top voxels between recursion and iteration for every participant.
# For every participant, there's a higher intersection than would be expected by chance
# between recursion and iteration conditions
results = compare_conditions(records, nulls, 'recursion', 'iteration')

participant_ids = results['participant_ids']
observed_vals   = results['observed_vals']
null_vals_list  = results['null_vals_list']
pvals           = results['pvals']
schaefers       = results['schaefers']
key_by_subset   = results['key_by_subset']
print(f"{len(participant_ids)} participants")

In [ ]:
# correct for multiple comparisons
reject, qvals, _, _ = multipletests(np.nan_to_num(pvals, nan=1.0), alpha=0.001, method='fdr_bh')
print(reject)

In [ ]:
plot_shared_voxels(participant_ids, observed_vals, null_vals_list,
                   title='Top-modeled Voxels Shared Between Recursion and Iteration')

In [ ]:
count_max = find_max(schaefers)
save_schaefer_maps(schaefers, atlases, vmax=count_max)

In [ ]:
# Group-level significance for the schaefer parcel tallies
# (see threshold_schaefers in _results_computation.py)
schaefers_thresholded, schaefer_qvals = threshold_schaefers(schaefers, nulls, participant_ids, key_by_subset, alpha=0.05)

for subset, qvals in schaefer_qvals.items():
    sig_parcels = np.where(qvals < 0.05)[0]
    print(f"{subset}: {len(sig_parcels)} parcels survive q<0.05 -> {sorted(sig_parcels.tolist())}")

In [ ]:
count_max = find_max(schaefers_thresholded)
save_schaefer_maps(schaefers_thresholded, atlases, vmax=count_max)